In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
from data import UncertainDataset, ds_single, ds_pressure

In [ ]:
from pathlib import Path

plots = Path("plots_lon_lat")

In [ ]:
import earthkit.plots
import earthkit.plots.quickmap
from matplotlib import pyplot as plt

def plot_data(data: xr.Dataset, plot: Path, colors = None):
    if colors:
        earthkit.plots.quickmap.plot(data.rename(dict(valid_time="time")), colors=colors)
    else:
         earthkit.plots.quickmap.plot(data.rename(dict(valid_time="time")))

    fig = plt.gcf()

    plot.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(plot, bbox_inches="tight")
    plt.show()

    plt.close(fig)  

In [ ]:
from tqdm import tqdm

In [ ]:
for key in tqdm(ds_single.mean):
    date = "2024-01-01T12:00:00"
    if key == "msl": # necessary because of an earthkit-plots bug
        plot_data(ds_single.mean[key].sel(valid_time=date), plots / "single" / f"lon_lat_{key}.png", colors ='viridis')
    else:
        plot_data(ds_single.mean[key].sel(valid_time=date), plots / "single" / f"lon_lat_{key}_{date}.png")


In [ ]:
for key in tqdm(ds_pressure.mean):
    for pressure in ds_pressure.mean[key].pressure_level.values:
        date = "2024-01-01T12:00:00"
        print(f"Pressure level {pressure} hPa:" )
        min_ = np.min(ds_pressure.mean[key].sel(pressure_level=pressure).sel(valid_time=date).values)
        max_ = np.max(ds_pressure.mean[key].sel(pressure_level=pressure).sel(valid_time=date).values)
        if min_ == max_:
            print(f"The data for {key} is constant {min_}.") 
        else:
            plot_data(ds_pressure.mean[key].sel(pressure_level=pressure).sel(valid_time=date), plots / "pressure" / f"lon_lat_{key}-{pressure}hPa_{date}.png")